# The Forensic Accounting Makeup Remover

### Investment Banking — Banking-AI


            ## Step 0 — Notebook Header

            **Prerequisites:** Basic Python, beginner familiarity with text analytics, and an understanding of financial statement footnotes.

            **What You Will Learn:**

            - Describe how suspicious footnote language can signal a possible forensic accounting issue.
- Compare a keyword baseline against a text-classification model for footnote risk screening.
- Construct a sentence-extraction workflow that surfaces the passages an analyst should review first.

            **Estimated time:** 45 minutes

            **Collection context:** Investment-banking notebooks that convert messy text and institutional context into reviewable risk signals.


## Step 1 — Investment-Banking Problem Introduction

**Why this problem matters:** Analysts cannot manually read every footnote across thousands of companies, so weak textual screening means suspicious cases stay hidden too long.

**Operational question:** Which footnotes deserve immediate forensic review, and which sentences explain the flag?

**Primary stakeholders:** forensic analysts, equity researchers, accounting specialists, and risk committees

**Decision supported:** Screen a large footnote universe for suspicious language and rank the passages that warrant human review.

**Comprehension check:** Before looking at the data, write one sentence explaining why a single phrase like `extended payment terms` might matter more than a bland positive sentence.

**Domain framing note:** This notebook is educational and synthetic. It demonstrates decision support for investment-banking and adjacent institutional workflows, not production approval or investment advice.


## Step 2 — Environment Setup

Use synthetic footnote text so the notebook remains fully shareable while still teaching the screening logic.


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)
RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)

print("Environment ready for a synthetic forensic-accounting workflow.")


## Step 3 — Data Creation & Context

We simulate footnote excerpts that include both ordinary accounting language and suspicious phrases tied to aggressive presentation choices.


In [ ]:
suspicious_phrases = [
    "extended payment terms",
    "supplier financing arrangement",
    "capitalized implementation costs",
    "one-time working capital release",
    "temporary covenant relief",
    "unusual receivable sale",
]
neutral_phrases = [
    "seasonal inventory build",
    "ordinary maintenance spending",
    "recurring service revenue",
    "stable customer retention",
    "normal hedging activity",
    "routine tax accrual",
]
rows = []

for _ in range(1400):
    suspicious_count = RNG.integers(0, 3)
    chosen_suspicious = RNG.choice(suspicious_phrases, size=suspicious_count, replace=False)
    chosen_neutral = RNG.choice(neutral_phrases, size=3, replace=False)
    footnote_text = ". ".join(list(chosen_neutral) + list(chosen_suspicious))
    risk_signal = suspicious_count * 0.95 + RNG.normal(0, 0.35)
    risk_label = "High risk" if risk_signal >= 1.25 else "Low risk"
    rows.append((footnote_text, suspicious_count, risk_label))

footnote_df = pd.DataFrame(rows, columns=["footnote_text", "suspicious_phrase_count", "risk_label"])
print(footnote_df.head(3).to_string(index=False))


## Step 4 — Exploratory Data Analysis

Check how many cases are high risk and how suspicious phrase counts vary before fitting a text model.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=footnote_df, x="risk_label", order=["Low risk", "High risk"], color="#C85A7C", ax=axes[0])
axes[0].set_title("Forensic Footnote Risk Labels")
axes[0].set_xlabel("Risk Label")
axes[0].set_ylabel("Count")

sns.boxplot(data=footnote_df, x="risk_label", y="suspicious_phrase_count", order=["Low risk", "High risk"], color="#C85A7C", ax=axes[1])
axes[1].set_title("Suspicious Phrase Count by Risk Label")
axes[1].set_xlabel("Risk Label")
axes[1].set_ylabel("Suspicious Phrase Count")

plt.tight_layout()
plt.show()
plt.close()


Alt-Text: The EDA shows that high-risk synthetic footnotes contain more suspicious phrases, which supports using text patterns as an initial forensic screen.


## Step 5 — Feature Engineering

We preserve interpretability by keeping the raw footnote text and adding a simple phrase-count feature for comparison.


In [ ]:
analysis_df = footnote_df.copy()
analysis_df["screen_text"] = analysis_df["footnote_text"]
print(analysis_df[["screen_text", "suspicious_phrase_count", "risk_label"]].head(3).to_string(index=False))


## Step 6 — Baseline Establishment

A natural baseline is a keyword screen: flag the footnote if it contains one of a few suspicious phrases.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    analysis_df["screen_text"],
    analysis_df["risk_label"],
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=analysis_df["risk_label"],
)

baseline_pred = X_test.apply(
    lambda text: "High risk" if any(phrase in text for phrase in suspicious_phrases[:4]) else "Low risk"
)
print(f"Baseline accuracy: {accuracy_score(y_test, baseline_pred):.3f}")


## Step 7 — Model Training & Evaluation

Train a text model that recognizes suspicious phrasing more flexibly than a fixed keyword list.


In [ ]:
forensic_model = Pipeline(
    [
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
        ("classifier", LogisticRegression(max_iter=1200)),
    ]
)
forensic_model.fit(X_train, y_train)
model_pred = forensic_model.predict(X_test)

print(f"Model accuracy: {accuracy_score(y_test, model_pred):.3f}")
print(classification_report(y_test, model_pred))


## Step 8 — Interpretability & Explainability

Explainability matters because analysts need to understand which phrases drove a flag before they escalate a company internally.


In [ ]:
vectorizer = forensic_model.named_steps["tfidf"]
classifier = forensic_model.named_steps["classifier"]
feature_names = vectorizer.get_feature_names_out()
positive_index = list(classifier.classes_).index("High risk")
coef = classifier.coef_[0]
if classifier.classes_[1] == "High risk":
    top_terms = feature_names[np.argsort(coef)[-10:]][::-1]
else:
    top_terms = feature_names[np.argsort(coef)[:10]]
print("Most suspicious learned terms:")
print(", ".join(top_terms))


## Step 9 — Operational Application

Use the model to surface both a probability-style flag and the exact sentences an analyst should read first.


In [ ]:
review_batch = analysis_df.sample(5, random_state=RANDOM_SEED).copy()
review_batch["high_risk_probability"] = forensic_model.predict_proba(review_batch["screen_text"])[:, positive_index]

def extract_flagged_sentences(text: str) -> str:
    sentences = [sentence.strip() for sentence in text.split(".") if sentence.strip()]
    flagged = [sentence for sentence in sentences if any(phrase in sentence for phrase in suspicious_phrases)]
    return " | ".join(flagged) if flagged else "no flagged sentence"

review_batch["flagged_sentences"] = review_batch["screen_text"].apply(extract_flagged_sentences)
print(review_batch[["screen_text", "high_risk_probability", "flagged_sentences"]].to_string(index=False))


            ## Step 10 — Summary & Reflection

            **What you accomplished:** You built a synthetic forensic-accounting screening workflow that spots suspicious footnote language and extracts the passages that deserve human review.

            **Limitations to note:**

            - The text is synthetic and much cleaner than real filings, which often contain boilerplate, nested references, and OCR issues.
- A phrase-level model cannot replace full accounting judgment or cross-statement analysis.
- Risk probabilities in this notebook are instructional scores, not forecasts of actual enforcement or share-price outcomes.

            **Ethical reflection:** Text screening can help prioritize scrutiny, but it should not become an automated accusation engine; flagged companies still require careful, evidence-based review.

            **Reflection questions:**

            1. Which additional filing section would you pair with footnotes to improve signal quality?
2. How would you handle companies that use suspicious language for benign operational reasons?
3. What evidence would justify escalating a low-probability but high-consequence case?


            ## References

            - Scikit-learn user guide: text feature extraction and logistic regression.
- Scenario inspiration: public forensic-accounting red-flag analysis and accounting-quality research.
- General literature on disclosure analytics and textual screening in financial statements.
